In [ ]:
import pandas as pd

df = pd.read_csv('../../data/pl_final.csv')

In [ ]:
df = df.sort_values('date')

df['home_xG_rolling'] = (
    df.groupby('home_team')['xG'].transform(lambda s: s.shift().rolling(5, min_periods=1).mean())
)

df['away_xG_rolling'] = (
    df.groupby('away_team')['xG.1'].transform(lambda s: s.shift().rolling(5, min_periods=1).mean())
)

df['xG_diff_rolling'] = df['home_xG_rolling'] - df['away_xG_rolling']

df['elo_diff'] = df['home_elo_before'] - df['away_elo_before']

df['goals_diff_rolling'] = df['home_goals_scored_rolling'] - df['away_goals_scored_rolling']
df['conceded_diff_rolling'] = df['home_goals_conceded_rolling'] - df['away_goals_conceded_rolling']

In [ ]:
df.dropna(inplace=True, axis=0)
df.columns

In [ ]:
# df = df[df['date'] >= '2025-08-01']

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    log_loss, brier_score_loss, confusion_matrix, classification_report
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = df.sort_values('date')

In [ ]:
X = df.drop(columns=['result', 'date', 'xG', 'xG.1', 'home_red_cards_rolling', 'away_red_cards_rolling', 'home_team', 'away_team'])
y = df['result']

In [ ]:
X.columns

In [ ]:
tscv = TimeSeriesSplit(n_splits=10)
rf_importances = []

In [ ]:
def multiclass_brier_score(y_true, y_prob):
    """
    Computes the multiclass Brier score (the mean squared error between
    one-hot labels and predicted probabilities).
    """
    n_classes = y_prob.shape[1]
    y_true_onehot = np.eye(n_classes)[y_true]  # One-hot encode labels
    return np.mean(np.sum((y_prob - y_true_onehot)**2, axis=1)) / n_classes

def evaluate_model(model, X_train, X_test, y_train, y_test, name="Model"):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)

    # --- Metrics ---
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average='weighted', zero_division=0)
    rec = recall_score(y_test, preds, average='weighted', zero_division=0)
    f1 = f1_score(y_test, preds, average='weighted', zero_division=0)
    ll = log_loss(y_test, probs)

    # Correct multiclass Brier score
    brier = multiclass_brier_score(y_test.to_numpy(), probs)


    print(f"\n=== {name} ===")
    print('accuracy:', acc)
    print('precision:', prec)
    print('recall:', rec)
    print('f1 score:', f1)
    print('log loss:', ll)
    print('brier score:', brier)

    # --- Classification Report ---
    print("\nClassification Report:")
    print(classification_report(
        y_test, preds, 
        target_names=["Draw", "Away", "Home"],
        zero_division=0
    ))

    # # --- Confusion Matrix ---
    cm = confusion_matrix(y_test, preds)
    # plt.figure(figsize=(6, 4))
    # sns.heatmap(
    #     cm, annot=True, fmt='d',
    #     xticklabels=["Draw", "Away", "Home"],
    #     yticklabels=["Draw", "Away", "Home"]
    # )
    # plt.title(f"Confusion Matrix: {name}")
    # plt.xlabel("Predicted")
    # plt.ylabel("True")
    # plt.show()

    metrics = {
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "log_loss": ll,
            "brier": brier,
            "confusion_matrix": cm
        }
    return metrics

In [ ]:
results = {
    "RandomForest": [],
    "XGBoost": [],
    "CatBoost": []
}

rf_importances = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    print(f"\n================ FOLD {fold+1} ================")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # --- 1. Random Forest ---
    rf = RandomForestClassifier(
        n_estimators=1500,
        max_depth=12,
        min_samples_split=4,
        random_state=42
    )

    m_rf = evaluate_model(rf, X_train, X_test, y_train, y_test, "Random Forest")
    results["RandomForest"].append(m_rf)
    rf_importances.append(rf.feature_importances_)
    
    # --- 2. XGBoost ---
    xgb = XGBClassifier(
        objective='multi:softprob',
        num_class=3,
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='mlogloss',
        tree_method='hist'
    )
    m_xgb = evaluate_model(xgb, X_train, X_test, y_train, y_test, "XGBoost")
    results["XGBoost"].append(m_xgb)

    # --- 3. CatBoost ---
    cat = CatBoostClassifier(
        loss_function='MultiClass',
        depth=7,
        iterations=1500,
        learning_rate=0.03,
        verbose=False
    )
    m_cat = evaluate_model(cat, X_train, X_test, y_train, y_test, "CatBoost")
    results["CatBoost"].append(m_cat)


In [ ]:
import pandas as pd

def summarize_results(results):
    summary = {}
    for model_name, folds in results.items():
        df = pd.DataFrame(folds)
        avg = df.mean(numeric_only=True)
        summary[model_name] = avg
        print(f"\n====== AVERAGE RESULTS: {model_name} ======")
        print(avg)
    return summary

summary = summarize_results(results)


In [ ]:
def average_confusion_matrix(model_name):
    cms = [r["confusion_matrix"] for r in results[model_name]]
    return sum(cms) / len(cms)

avg_cm_rf = average_confusion_matrix("RandomForest")
avg_cm_xgb = average_confusion_matrix("XGBoost")
avg_cm_cat = average_confusion_matrix("CatBoost")


In [ ]:
labels = ['Draw', 'Loss', 'Win']
avg_cm_rf_norm = avg_cm_rf / avg_cm_rf.sum(axis=1, keepdims=True)

plt.figure(figsize=(10,4))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(avg_cm_rf, annot=True, fmt=".1f", cmap="Blues",
            xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title("Avg Confusion Matrix (Counts)")
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(avg_cm_rf_norm, annot=True, fmt=".2f", cmap="Greens",
            xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title("Avg Confusion Matrix (Normalized)")
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Average across folds
rf_importances = np.array(rf_importances)
avg_importances = rf_importances.mean(axis=0)

# Put in a DataFrame for readability
fi_df = pd.DataFrame({
    "feature": X.columns,
    "importance": avg_importances
}).sort_values("importance", ascending=False)

In [ ]:
fi_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Sort values for clearer plotting
fi_sorted = fi_df.sort_values("importance", ascending=False).head(12)

plt.figure(figsize=(8,5))
sns.barplot(
    data=fi_sorted,
    x="importance",
    y="feature",
    palette='viridis'
)

plt.title("Random Forest Most Important Features (Averaged Across Folds)", fontsize=16)
plt.xlabel("Importance", fontsize=12)
plt.ylabel("Feature", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
home_baseline_acc = (y_test == 2).mean()
print(f"Baseline (Always Home Win): {home_baseline_acc:.2f}")


In [ ]:
most_common = y_train.value_counts().idxmax()
print(most_common)
freq_baseline_acc = (y_test == most_common).mean()
print(f"Baseline (Most Frequent Class = {most_common}): {freq_baseline_acc:.2f}")


In [ ]:
results["RandomForest"]

In [ ]:
import matplotlib.pyplot as plt

def plot_metrics_over_folds(results_dict, model_name="RandomForest"):
    # Only plot scalar metrics per fold; confusion_matrix is a 2D array and
    # doesn't belong on a per-fold line plot.
    df = pd.DataFrame(results_dict).drop(columns=["confusion_matrix"])
    folds = range(1, len(df) + 1)

    plt.figure(figsize=(14, 8))

    for i, metric in enumerate(df.columns):
        plt.subplot(2, 3, i + 1)
        plt.plot(folds, df[metric], marker='o')
        plt.title(metric)
        plt.xlabel("Fold")
        plt.ylabel(metric)
        plt.grid(True)

    plt.suptitle(f"Metrics over folds — {model_name}", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

# Use it:
plot_metrics_over_folds(results["RandomForest"], "RandomForest")


In [ ]:
def compare_models_metrics(results, metric):
    plt.figure(figsize=(12,6))

    for model_name, model_results in results.items():
        df = pd.DataFrame(model_results)
        plt.plot(df[metric], marker='o', label=model_name)

    plt.title(f"{metric} across folds")
    plt.ylim(0.2, 0.8)
    plt.xlabel("Fold")
    plt.ylabel(metric)
    plt.legend()
    plt.grid(True)
    plt.show()

compare_models_metrics(results, "accuracy")
compare_models_metrics(results, "log_loss")
compare_models_metrics(results, "brier")


In [ ]:
rf = RandomForestClassifier(
    n_estimators=1500,
    max_depth=12,
    min_samples_split=4,
    random_state=42
)

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

final_rf = CalibratedClassifierCV(rf, method='isotonic', cv=5)
final_rf.fit(X, y)

In [ ]:
# Predictions on training set
train_preds = final_rf.predict(X)

# Training accuracy
train_acc = accuracy_score(y, train_preds)
print("Training accuracy:", train_acc)

train_probs = final_rf.predict_proba(X)

print("Training precision:", precision_score(y, train_preds, average='weighted'))
print("Training recall:", recall_score(y, train_preds, average='weighted'))
print("Training f1:", f1_score(y, train_preds, average='weighted'))
print("Training log loss:", log_loss(y, train_probs))


In [ ]:
import joblib
joblib.dump(final_rf, "../../models/random_forest_model.pkl", compress=3)

feature_names = X.columns.tolist()
features_path = '../../models/feature_names.pkl'
joblib.dump(feature_names, features_path, compress=3)

In [ ]:
# --- 2. XGBoost ---
xgb = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    n_estimators=1500,
    learning_rate=0.03,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    tree_method='hist'
)

# --- 3. CatBoost ---
cat = CatBoostClassifier(
    loss_function='MultiClass',
    depth=7,
    iterations=1500,
    learning_rate=0.03,
    verbose=False
)

In [ ]:
final_xgb = CalibratedClassifierCV(xgb, method='isotonic', cv=5)
final_xgb.fit(X, y)

In [ ]:
# Predictions on training set
train_preds = final_xgb.predict(X)

# Training accuracy
train_acc = accuracy_score(y, train_preds)
print("Training accuracy:", train_acc)

train_probs = final_xgb.predict_proba(X)

print("Training precision:", precision_score(y, train_preds, average='weighted'))
print("Training recall:", recall_score(y, train_preds, average='weighted'))
print("Training f1:", f1_score(y, train_preds, average='weighted'))
print("Training log loss:", log_loss(y, train_probs))
joblib.dump(final_xgb, "../../models/xgboost_model.pkl", compress=3)

In [ ]:
final_cat = CalibratedClassifierCV(cat, method='isotonic', cv=5)
final_cat.fit(X, y)

In [ ]:
train_preds = final_cat.predict(X)
train_acc = accuracy_score(y, train_preds)
print("Training accuracy:", train_acc)
train_probs = final_cat.predict_proba(X)
print("Training precision:", precision_score(y, train_preds, average='weighted'))
print("Training recall:", recall_score(y, train_preds, average='weighted'))
print("Training f1:", f1_score(y, train_preds, average='weighted'))
print("Training log loss:", log_loss(y, train_probs))
joblib.dump(final_cat, "../../models/catboost_model.pkl", compress=3)

In [ ]:
from sklearn.calibration import calibration_curve

def plot_calibration(model, X_test, y_test, class_index, class_name):
    probs = model.predict_proba(X_test)[:, class_index]
    true_labels = (y_test == class_index).astype(int)

    prob_true, prob_pred = calibration_curve(true_labels, probs, n_bins=10)

    plt.plot(prob_pred, prob_true, marker='o')
    plt.plot([0,1], [0,1], linestyle='--')
    plt.title(f"Calibration Curve: {class_name}")
    plt.xlabel("Predicted probability")
    plt.ylabel("True probability")
    plt.show()

In [ ]:
plot_calibration(final_rf, X_test, y_test, 0, "Draw")
plot_calibration(final_rf, X_test, y_test, 1, "Away")
plot_calibration(final_rf, X_test, y_test, 2, "Home")

In [ ]:
plot_calibration(final_xgb, X_test, y_test, 0, "Draw")
plot_calibration(final_xgb, X_test, y_test, 1, "Away")
plot_calibration(final_xgb, X_test, y_test, 2, "Home")

In [ ]:
plot_calibration(final_cat, X_test, y_test, 0, "Draw")
plot_calibration(final_cat, X_test, y_test, 1, "Away")
plot_calibration(final_cat, X_test, y_test, 2, "Home")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Value counts
result_counts = df['result'].value_counts().sort_index()
labels = ['Draw', 'Loss', 'Win']

plt.figure(figsize=(12,6))
sns.barplot(x=labels, y=result_counts.values , palette='viridis')
plt.title('Distribution of Match Outcomes (Home Team Perspective)')
plt.xlabel('Result')
plt.ylabel('Number of Matches')
plt.show()


In [ ]:
import joblib
rf = joblib.load('../../models/random_forest_model.pkl')

import matplotlib.pyplot as plt

probs = rf.predict_proba(X)[:, 2]   # class index for home win

plt.hist(probs, bins=20)
plt.xlabel("Predicted probability of Home Win")
plt.ylabel("Count")
plt.title("Distribution of Predicted Probabilities – Home Win")
plt.show()



In [ ]:
import seaborn as sns
import pandas as pd

probs = rf.predict_proba(X)

df = pd.DataFrame({
    "prob_draw": probs[:, 0],
    "prob_away": probs[:, 1],
    "prob_home": probs[:, 2],
})

sns.kdeplot(df["prob_home"], label="Home")
sns.kdeplot(df["prob_away"], label="Away")
sns.kdeplot(df["prob_draw"], label="Draw")
plt.title("Predicted Probability Density for Each Outcome")
plt.xlim(0, 1)
plt.xlabel("Predicted Probability")
plt.legend()
plt.show()


In [ ]:
%pip install python-ternary
import ternary

figure, tax = ternary.figure(scale=1)
tax.boundary()
tax.gridlines(color="black", multiple=0.1)

points = list(map(tuple, probs))
tax.scatter(points, s=2)

tax.left_corner_label("Draw")
tax.right_corner_label("Away")
tax.top_corner_label("Home")
tax.set_title("Ternary Plot of Prediction Probabilities")

plt.show()


## Honest held-out evaluation for the dashboard

The models above (`final_rf`, `final_xgb`, `final_cat`) are fit on the full dataset for production use, so scoring them against any slice of that same data would be data leakage. This cell trains fresh copies on a strict chronological 80/20 split (train on the first 80% of matches by date, evaluate only on the untouched last 20%) and caches the results for the Streamlit performance page.

In [ ]:
import json

# Self-contained: re-derive the feature-engineered dataframe here rather than
# reusing the notebook-level `df`/`X`/`y`, since a later cell above reassigns
# `df` to an unrelated probabilities table.
df_eval = pd.read_csv("../../data/pl_final.csv")
df_eval = df_eval.sort_values("date")

df_eval["home_xG_rolling"] = df_eval.groupby("home_team")["xG"].transform(lambda s: s.shift().rolling(5, min_periods=1).mean())
df_eval["away_xG_rolling"] = df_eval.groupby("away_team")["xG.1"].transform(lambda s: s.shift().rolling(5, min_periods=1).mean())
df_eval["xG_diff_rolling"] = df_eval["home_xG_rolling"] - df_eval["away_xG_rolling"]
df_eval["elo_diff"] = df_eval["home_elo_before"] - df_eval["away_elo_before"]
df_eval["goals_diff_rolling"] = df_eval["home_goals_scored_rolling"] - df_eval["away_goals_scored_rolling"]
df_eval["conceded_diff_rolling"] = df_eval["home_goals_conceded_rolling"] - df_eval["away_goals_conceded_rolling"]

df_eval = df_eval.dropna(axis=0)
df_eval = df_eval.sort_values("date")

X_eval_full = df_eval.drop(columns=["result", "date", "xG", "xG.1", "home_red_cards_rolling", "away_red_cards_rolling", "home_team", "away_team"])
y_eval_full = df_eval["result"]

train_size_eval = int(len(df_eval) * 0.8)
X_train_eval, X_test_eval = X_eval_full.iloc[:train_size_eval], X_eval_full.iloc[train_size_eval:]
y_train_eval, y_test_eval = y_eval_full.iloc[:train_size_eval], y_eval_full.iloc[train_size_eval:]

def multiclass_brier_score_eval(y_true, y_prob):
    n_classes = y_prob.shape[1]
    y_true_onehot = np.eye(n_classes)[y_true]
    return np.mean(np.sum((y_prob - y_true_onehot) ** 2, axis=1)) / n_classes

def evaluate_holdout(name, model):
    calib = CalibratedClassifierCV(model, method="isotonic", cv=5)
    calib.fit(X_train_eval, y_train_eval)
    preds = calib.predict(X_test_eval)
    probs = calib.predict_proba(X_test_eval)
    metrics = {
        "accuracy": accuracy_score(y_test_eval, preds),
        "precision": precision_score(y_test_eval, preds, average="weighted", zero_division=0),
        "recall": recall_score(y_test_eval, preds, average="weighted", zero_division=0),
        "f1_score": f1_score(y_test_eval, preds, average="weighted", zero_division=0),
        "log_loss": log_loss(y_test_eval, probs),
        "brier_score": multiclass_brier_score_eval(y_test_eval.to_numpy(), probs),
        "confusion_matrix": confusion_matrix(y_test_eval, preds).tolist(),
        "test_size": int(len(y_test_eval)),
        "train_size": int(len(y_train_eval)),
    }
    return metrics, calib

rf_eval = RandomForestClassifier(n_estimators=1500, max_depth=12, min_samples_split=4, random_state=42)
xgb_eval = XGBClassifier(objective="multi:softprob", num_class=3, n_estimators=1500, learning_rate=0.03,
                          max_depth=7, subsample=0.8, colsample_bytree=0.8, eval_metric="mlogloss", tree_method="hist")
cat_eval = CatBoostClassifier(loss_function="MultiClass", depth=7, iterations=1500, learning_rate=0.03, verbose=False)

rf_holdout_metrics, rf_holdout_model = evaluate_holdout("RandomForest", rf_eval)
xgb_holdout_metrics, _ = evaluate_holdout("XGBoost", xgb_eval)
cat_holdout_metrics, _ = evaluate_holdout("CatBoost", cat_eval)

all_holdout_metrics = {
    "random_forest": rf_holdout_metrics,
    "xgboost": xgb_holdout_metrics,
    "catboost": cat_holdout_metrics,
    "_meta": {
        "total_matches": int(len(df_eval)),
        "train_size": int(len(X_train_eval)),
        "test_size": int(len(X_test_eval)),
        "test_start_date": str(df_eval["date"].iloc[train_size_eval]),
        "test_end_date": str(df_eval["date"].iloc[-1]),
        "feature_count": int(X_eval_full.shape[1]),
        "class_encoding": {"0": "Draw", "1": "Away Win", "2": "Home Win"},
    },
}

with open("../../models/performance_metrics.json", "w") as f:
    json.dump(all_holdout_metrics, f, indent=2)

predictions_df = df_eval.iloc[train_size_eval:][["date", "home_team", "away_team"]].copy()
predictions_df["actual"] = y_test_eval.to_numpy()
predictions_df["predicted"] = rf_holdout_model.predict(X_test_eval)
predictions_df.to_csv("../../models/performance_test_predictions.csv", index=False)

print("Held-out accuracy (Random Forest):", rf_holdout_metrics["accuracy"])
